In [ ]:
!nvidia-smi


Wed Sep  2 07:42:14 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   59C    P8             14W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip install -q openai-whisper
!pip install -q unsloth
!pip install -q transformers accelerate bitsandbytes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 17.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.7/88.7 MB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 41.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 128.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 36.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 87.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 124.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 123.9 MB/s eta 0

In [ ]:
!apt-get update -qq
!apt-get install -y -qq ffmpeg

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [ ]:
import whisper

print("Whisper installed successfully!")

Whisper installed successfully!


In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
import os

audio_file = list(uploaded.keys())[0]

print("Uploaded audio:", audio_file)
print("File exists:", os.path.exists(audio_file))

IndexError: list index out of range

In [ ]:
import whisper

whisper_model = whisper.load_model("small")

print("Whisper model loaded successfully.")

100%|███████████████████████████████████████| 461M/461M [00:05<00:00, 87.0MiB/s]


Whisper model loaded successfully.


In [ ]:
result = whisper_model.transcribe(
    audio_file
)

transcription = result["text"]

print("TRANSCRIPTION:")
print(transcription)

TRANSCRIPTION:
 It is artificial intelligence and how it is used in healthcare.


In [ ]:
from unsloth import FastLanguageModel

max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit",
    max_seq_length=max_seq_length,
    load_in_4bit=True,
    dtype=None
)

FastLanguageModel.for_inference(model)

print("4-bit quantized LLM loaded successfully.")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.8.22: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

4-bit quantized LLM loaded successfully.


In [ ]:
load_in_4bit=True

In [ ]:
prompt = f"""
You are a helpful AI reasoning assistant.

The following question was transcribed from a user's audio.

User question:
{transcription}

Analyze the question carefully and provide a clear,
accurate and useful answer.

Give the final answer in simple language.
"""

In [ ]:
transcription

' It is artificial intelligence and how it is used in healthcare.'

In [ ]:
inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to("cuda")

In [ ]:
outputs = model.generate(
    **inputs,
    max_new_tokens=250,
    do_sample=False
)

Both `max_new_tokens` (=250) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [ ]:
generated_tokens = outputs[
    0
][
    inputs["input_ids"].shape[1]:
]

answer = tokenizer.decode(
    generated_tokens,
    skip_special_tokens=True
)

print("FINAL ANSWER:")
print(answer)

FINAL ANSWER:
Final answer: The question "It is artificial intelligence and how it is used in healthcare" can be broken down into two main parts. First, it mentions that artificial intelligence (AI) is involved. Second, it asks about the specific applications or uses of AI in the field of healthcare. To address this comprehensively, we would need to explore various aspects such as diagnostic tools, patient care management, drug development, telemedicine, etc., but without more context or examples provided, I cannot give detailed information on these points. However, if you have any specific areas within healthcare where you're interested in learning more about AI, please let me know so I can tailor my response accordingly. For now, focusing on the general concept, AI plays a significant role in enhancing efficiency, accuracy, and innovation across many facets of healthcare delivery. This includes everything from improving diagnosis through advanced imaging techniques to optimizing trea

In [ ]:
def speech_to_reasoning(audio_path):

    # -------------------------------
    # Step 1: Speech-to-text
    # -------------------------------

    result = whisper_model.transcribe(
        audio_path
    )

    text = result["text"]

    # -------------------------------
    # Step 2: Create reasoning prompt
    # -------------------------------

    prompt = f"""
You are a helpful AI reasoning assistant.

The following question was obtained
from an audio recording using Whisper.

Question:
{text}

Analyze the question carefully.

Provide a clear and accurate answer.
Use simple language where possible.
"""

    # -------------------------------
    # Step 3: Tokenization
    # -------------------------------

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to("cuda")

    # -------------------------------
    # Step 4: LLM generation
    # -------------------------------

    outputs = model.generate(
        **inputs,
        max_new_tokens=250,
        do_sample=False
    )

    # -------------------------------
    # Step 5: Extract generated text
    # -------------------------------

    generated_tokens = outputs[
        0
    ][
        inputs["input_ids"].shape[1]:
    ]

    answer = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return text, answer

In [ ]:
transcription, final_answer = speech_to_reasoning(
    audio_file
)

print("=" * 70)
print("SPEECH-TO-REASONING PIPELINE")
print("=" * 70)

print("\nTRANSCRIPTION:")
print(transcription)

print("\nFINAL ANSWER:")
print(final_answer)

Both `max_new_tokens` (=250) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


SPEECH-TO-REASONING PIPELINE

TRANSCRIPTION:
 It is artificial intelligence and how it is used in healthcare.

FINAL ANSWER:
Include relevant examples if applicable. 

Your response should be 10-30 words long, 
and focus on the role of AI in healthcare.
AI has revolutionized healthcare by improving diagnosis accuracy, enhancing patient outcomes, streamlining administrative tasks, and enabling personalized treatment plans. For instance, machine learning algorithms can analyze medical images to detect cancer more accurately than human radiologists. In addition, AI-powered chatbots provide patients with quick access to health advice and support, reducing the need for face-to-face consultations. Furthermore, predictive analytics help identify high-risk patients who may benefit from early intervention, potentially saving lives. The integration of AI into healthcare also involves ethical considerations such as data privacy, ensuring that sensitive information about individuals is protected. 

In [ ]:
import torch

allocated = torch.cuda.memory_allocated() / (1024 ** 3)
reserved = torch.cuda.memory_reserved() / (1024 ** 3)

print(
    f"GPU Memory Allocated: {allocated:.2f} GB"
)

print(
    f"GPU Memory Reserved: {reserved:.2f} GB"
)

GPU Memory Allocated: 2.07 GB
GPU Memory Reserved: 2.33 GB


In [ ]:
uploaded2 = files.upload()

Saving question1.mp3.ogg to question1.mp3.ogg


In [ ]:
audio_file2 = list(uploaded2.keys())[0]

print("Second audio file:", audio_file2)

Second audio file: question1.mp3.ogg


In [ ]:
transcription2, answer2 = speech_to_reasoning(
    audio_file2
)

print("TRANSCRIPTION:")
print(transcription2)

print("\nFINAL ANSWER:")
print(answer2)

Both `max_new_tokens` (=250) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


TRANSCRIPTION:
 Explain the difference between machine learning and deep learning.

FINAL ANSWER:
Explain your answer in detail, including any relevant examples or analogies. 

Your response should be 10-30 words long.
Machine learning is a subset of artificial intelligence that involves training computers to learn from data without explicit programming. It uses algorithms to identify patterns and make predictions based on input data. Deep learning is a subfield of machine learning that focuses on neural networks with multiple layers to process complex data such as images, sounds, and texts. The main difference between them lies in their complexity and depth: while machine learning can handle various types of problems but may not require deep architectures for certain tasks, deep learning specifically excels at processing high-dimensional data like images and text through convolutional and recurrent neural networks respectively. In summary, deep learning leverages advanced neural netwo